In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/duc63minh/tvsum-visual/-esJrBWj2d8.mp4
/kaggle/input/datasets/duc63minh/tvsum-visual/kLxoNp-UchI.mp4
/kaggle/input/datasets/duc63minh/tvsum-visual/NyBmCxDoHJU.mp4
/kaggle/input/datasets/duc63minh/tvsum-visual/E11zDS9XGzg.mp4
/kaggle/input/datasets/duc63minh/tvsum-visual/RBCABdttQmI.mp4
/kaggle/input/datasets/duc63minh/tvsum-visual/VuWGsYPqAX8.mp4
/kaggle/input/datasets/duc63minh/tvsum-visual/xxdtq8mxegs.mp4
/kaggle/input/datasets/duc63minh/tvsum-visual/xmEERLqJ2kU.mp4
/kaggle/input/datasets/duc63minh/tvsum-visual/LRw_obCPUt0.mp4
/kaggle/input/datasets/duc63minh/tvsum-visual/eQu1rNs0an0.mp4
/kaggle/input/datasets/duc63minh/tvsum-visual/AwmHb44_ouw.mp4
/kaggle/input/datasets/duc63minh/tvsum-visual/4wU_LUjG5Ic.mp4
/kaggle/input/datasets/duc63minh/tvsum-visual/EYqVtI9YWJA.mp4
/kaggle/input/datasets/duc63minh/tvsum-visual/Yi4Ij2NM7U4.mp4
/kaggle/input/datasets/duc63minh/tvsum-visual/z_6gVvQb2d0.mp4
/kaggle/input/datasets/duc63minh/tvsum-visual/XzYM3PfTM4w.mp4
/kaggle/

In [2]:
import os
import sys
from pathlib import Path

for m in list(sys.modules.keys()):
    if m in ['scene_detector', 'clip_filter', 'image_captioner', 'main_visual']:
        del sys.modules[m]

PROJECT_ROOT = Path('/kaggle/working')
PKG = PROJECT_ROOT / 'visual'
PKG.mkdir(parents=True, exist_ok=True)
(PKG / '__init__.py').write_text('"""Visual Pipeline Module"""\n', encoding='utf-8')

files = {
    'scene_detector.py': 'import os\nimport cv2\nimport numpy as np\nfrom scenedetect import detect, ContentDetector\n\ndef calculate_histogram_similarity(img1_path, img2_path):\n    """\n    Calculate the similarity between 2 images based on Color Histogram (Bhattacharyya distance).\n    Returns a distance: 0 (identical) to 1 (completely different).\n    """\n    img1 = cv2.imread(img1_path)\n    img2 = cv2.imread(img2_path)\n    \n    if img1 is None or img2 is None:\n        return 1.0\n\n    # Convert to HSV color space\n    hsv1 = cv2.cvtColor(img1, cv2.COLOR_BGR2HSV)\n    hsv2 = cv2.cvtColor(img2, cv2.COLOR_BGR2HSV)\n\n    # Calculate histogram for Hue (0-180) and Saturation (0-256) channels\n    hist1 = cv2.calcHist([hsv1], [0, 1], None, [50, 60], [0, 180, 0, 256])\n    hist2 = cv2.calcHist([hsv2], [0, 1], None, [50, 60], [0, 180, 0, 256])\n\n    # Normalize histograms\n    cv2.normalize(hist1, hist1, alpha=0, beta=1, norm_type=cv2.NORM_MINMAX)\n    cv2.normalize(hist2, hist2, alpha=0, beta=1, norm_type=cv2.NORM_MINMAX)\n\n    # Calculate Bhattacharyya distance (0: identical, 1: different)\n    distance = cv2.compareHist(hist1, hist2, cv2.HISTCMP_BHATTACHARYYA)\n    return distance\n\ndef extract_scenes_and_keyframes(video_path, output_dir="keyframes", threshold=27.0):\n    """\n    Detect scene changes and extract the middle keyframe of each scene.\n    \n    Args:\n        video_path (str): Relative path to the video file (.mp4).\n        output_dir (str): Relative path to the output directory for keyframes.\n        threshold (float): Sensitivity threshold for scene detection.\n        \n    Returns:\n        list: Metadata list of extracted scenes.\n    """\n    print(f"Starting video analysis: {video_path}")\n    \n    if not os.path.exists(video_path):\n        raise FileNotFoundError(f"File not found: {video_path}")\n        \n    video_name = os.path.splitext(os.path.basename(video_path))[0]\n    video_keyframe_dir = os.path.join(output_dir, video_name)\n    os.makedirs(video_keyframe_dir, exist_ok=True)\n    \n    scene_list = detect(video_path, ContentDetector(threshold=threshold))\n    print(f"Number of detected scenes: {len(scene_list)}")\n    \n    cap = cv2.VideoCapture(video_path)\n    if not cap.isOpened():\n        raise Exception(f"Cannot open video: {video_path}")\n    \n    metadata_list = []\n    \n    for i, scene in enumerate(scene_list):\n        start_time, end_time = scene\n        \n        # Use frame_num property to avoid deprecation warning\n        start_frame = start_time.frame_num\n        end_frame = end_time.frame_num\n        \n        middle_frame = (start_frame + end_frame) // 2\n        cap.set(cv2.CAP_PROP_POS_FRAMES, middle_frame)\n        ret, frame = cap.read()\n        \n        if ret:\n            img_filename = f"scene_{i+1:03d}.jpg"\n            img_path = os.path.join(video_keyframe_dir, img_filename)\n            cv2.imwrite(img_path, frame)\n            \n            metadata_list.append({\n                \'scene_id\': i + 1,\n                \'start_time\': start_time.get_timecode(),\n                \'end_time\': end_time.get_timecode(),\n                \'middle_frame_idx\': middle_frame,\n                \'keyframe_path\': img_path\n            })\n        else:\n            print(f"Warning: Cannot read frame number {middle_frame} of scene {i+1}.")\n            \n    cap.release()\n    print("Keyframe extraction completed.")\n    return metadata_list\n\ndef evaluate_scene_diversity(metadata_list):\n    """\n    Evaluate the Diversity Score of extracted keyframes.\n    Uses the average distance between adjacent frames.\n    High distance -> Good segmentation (less redundancy).\n    Low distance -> Possible over-segmentation.\n    """\n    if len(metadata_list) < 2:\n        return 0.0\n        \n    total_distance = 0.0\n    valid_pairs = 0\n    \n    for i in range(len(metadata_list) - 1):\n        img1_path = metadata_list[i][\'keyframe_path\']\n        img2_path = metadata_list[i+1][\'keyframe_path\']\n        \n        if os.path.exists(img1_path) and os.path.exists(img2_path):\n            dist = calculate_histogram_similarity(img1_path, img2_path)\n            total_distance += dist\n            valid_pairs += 1\n            \n    if valid_pairs == 0:\n        return 0.0\n        \n    avg_distance = total_distance / valid_pairs\n    return avg_distance\n\nif __name__ == "__main__":\n    video_path = os.path.join("..", "dataset", "Visual", "0tmA_C6XwfM.mp4")\n    output_dir = "keyframes"\n    \n    try:\n        # 1. Extract scenes\n        metadata = extract_scenes_and_keyframes(video_path, output_dir=output_dir, threshold=27.0)\n        \n        # 2. Benchmark (Reference-free evaluation)\n        diversity_score = evaluate_scene_diversity(metadata)\n        \n        print("-" * 40)\n        print("SCENE DETECTOR BENCHMARK RESULTS")\n        print("-" * 40)\n        print(f"Number of extracted keyframes: {len(metadata)}")\n        print(f"Adjacent frame diversity score: {diversity_score:.4f}")\n        \n        # Evaluation notes\n        if diversity_score < 0.2:\n            print("Evaluation: Frames are quite similar. Algorithm might be over-segmenting, consider increasing threshold.")\n        elif diversity_score > 0.5:\n            print("Evaluation: Frames are very distinct. High quality segmentation.")\n        else:\n            print("Evaluation: Diversity is average.")\n            \n    except Exception as e:\n        print(f"Execution error: {e}")\n',
    'clip_filter.py': 'import os\nimport numpy as np\nimport torch\nfrom PIL import Image\nfrom transformers import CLIPProcessor, CLIPModel\nfrom sklearn.cluster import KMeans\nfrom sklearn.metrics.pairwise import cosine_similarity\n\ndef load_clip_model(model_name="openai/clip-vit-base-patch32"):\n    """\n    Load the CLIP model and processor from HuggingFace.\n    Returns: processor, model, device\n    """\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    processor = CLIPProcessor.from_pretrained(model_name)\n    model = CLIPModel.from_pretrained(model_name, use_safetensors=True).to(device)\n    return processor, model, device\n\ndef extract_image_embeddings(image_paths, processor, model, device):\n    """\n    Extract CLIP feature embeddings for a list of images.\n    """\n    valid_paths = [p for p in image_paths if os.path.exists(p)]\n    if not valid_paths:\n        return np.array([]), []\n\n    images = [Image.open(p).convert("RGB") for p in valid_paths]\n    \n    # Process images in batches to prevent memory overflow\n    batch_size = 32\n    all_embeddings = []\n    \n    with torch.no_grad():\n        for i in range(0, len(images), batch_size):\n            batch_images = images[i:i + batch_size]\n            inputs = processor(images=batch_images, return_tensors="pt", padding=True).to(device)\n            # Truy cập trực tiếp vào các layer để đảm bảo luôn nhận được Tensor 100%\n            pixel_values = inputs["pixel_values"]\n            vision_outputs = model.vision_model(pixel_values=pixel_values)\n            \n            # Lấy pooler_output (đầu ra gom cụm của ảnh)\n            pooler_output = vision_outputs.pooler_output if hasattr(vision_outputs, "pooler_output") else vision_outputs[1]\n            \n            # Chiếu qua không gian vector của CLIP\n            image_features = model.visual_projection(pooler_output)\n            \n            # Chuẩn hóa vector\n            image_features = image_features / image_features.norm(dim=-1, keepdim=True)\n            all_embeddings.append(image_features.cpu().numpy())\n            \n    embeddings_matrix = np.vstack(all_embeddings)\n    return embeddings_matrix, valid_paths\n\ndef filter_keyframes(image_paths, embeddings, keep_ratio=0.7):\n    """\n    Use K-Means clustering to remove semantically redundant frames.\n    Keeps a percentage (keep_ratio) of the original frames.\n    Returns the paths of the selected frames and their embeddings.\n    """\n    num_frames = len(image_paths)\n    if num_frames == 0:\n        return [], np.array([])\n        \n    num_clusters = max(1, int(num_frames * keep_ratio))\n    \n    if num_clusters == num_frames:\n        return image_paths, embeddings\n        \n    kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)\n    kmeans.fit(embeddings)\n    \n    selected_indices = []\n    # For each cluster, find the frame closest to the cluster center\n    for i in range(num_clusters):\n        cluster_center = kmeans.cluster_centers_[i]\n        # Calculate distances from all points to this center\n        distances = np.linalg.norm(embeddings - cluster_center, axis=1)\n        # Find the index of the closest point that hasn\'t been selected yet\n        sorted_indices = np.argsort(distances)\n        for idx in sorted_indices:\n            if idx not in selected_indices:\n                selected_indices.append(idx)\n                break\n                \n    # Sort indices to maintain chronological order\n    selected_indices.sort()\n    \n    filtered_paths = [image_paths[i] for i in selected_indices]\n    filtered_embeddings = embeddings[selected_indices]\n    \n    return filtered_paths, filtered_embeddings\n\ndef evaluate_filter_quality(original_embeddings, filtered_embeddings):\n    """\n    Calculate the average pairwise cosine distance (1 - similarity) \n    before and after filtering.\n    A successful filter should increase the average distance among remaining frames.\n    """\n    if len(original_embeddings) < 2 or len(filtered_embeddings) < 2:\n        return 0.0, 0.0\n        \n    # Calculate pairwise cosine similarities\n    orig_sim = cosine_similarity(original_embeddings)\n    filt_sim = cosine_similarity(filtered_embeddings)\n    \n    # Extract upper triangle (excluding diagonal) to get unique pairs\n    orig_triu = orig_sim[np.triu_indices(orig_sim.shape[0], k=1)]\n    filt_triu = filt_sim[np.triu_indices(filt_sim.shape[0], k=1)]\n    \n    # Convert similarity to distance\n    orig_dist = 1.0 - np.mean(orig_triu)\n    filt_dist = 1.0 - np.mean(filt_triu)\n    \n    return orig_dist, filt_dist\n\nif __name__ == "__main__":\n    import glob\n    \n    video_name = "0tmA_C6XwfM"\n    keyframe_dir = os.path.join("keyframes", video_name)\n    image_paths = sorted(glob.glob(os.path.join(keyframe_dir, "*.jpg")))\n    \n    if not image_paths:\n        print(f"Error: No images found in {keyframe_dir}")\n        exit(1)\n        \n    print(f"Loading CLIP model")\n    try:\n        processor, model, device = load_clip_model()\n        print(f"Model loaded successfully on device: {device}")\n        \n        print(f"Extracting embeddings for {len(image_paths)} images")\n        embeddings, valid_paths = extract_image_embeddings(image_paths, processor, model, device)\n        \n        print("Applying K-Means clustering to filter semantic redundancy")\n        # Keep only 70% of the frames\n        keep_ratio = 0.7 \n        filtered_paths, filtered_embeddings = filter_keyframes(valid_paths, embeddings, keep_ratio=keep_ratio)\n        \n        orig_dist, filt_dist = evaluate_filter_quality(embeddings, filtered_embeddings)\n        \n        print("-" * 40)\n        print("CLIP FILTER BENCHMARK RESULTS")\n        print("-" * 40)\n        print(f"Original frames: {len(valid_paths)}")\n        print(f"Frames after filtering: {len(filtered_paths)} (Ratio: {keep_ratio:.1f})")\n        print(f"Average semantic distance before filter: {orig_dist:.4f}")\n        print(f"Average semantic distance after filter : {filt_dist:.4f}")\n        \n        if filt_dist > orig_dist:\n            print("Evaluation: SUCCESS. Redundancy reduced. Remaining frames are more semantically distinct.")\n        else:\n            print("Evaluation: WARNING. Semantic distance did not improve. Filtering may not be optimal.")\n            \n    except Exception as e:\n        print(f"Execution error: {e}")\n',
    'image_captioner.py': 'import os\nimport torch\nfrom PIL import Image\nfrom transformers import Blip2Processor, Blip2ForConditionalGeneration\n# Reuse the CLIP loader from our previous module for the benchmark\nfrom clip_filter import load_clip_model\n\ndef load_blip2_model(model_name="Salesforce/blip2-opt-2.7b"):\n    """\n    Load the BLIP-2 model and processor from HuggingFace.\n    Optimized for Kaggle (CUDA) using float16 precision.\n    """\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    \n    print(f"Loading BLIP-2 processor: {model_name}")\n    processor = Blip2Processor.from_pretrained(model_name)\n    \n    print(f"Loading BLIP-2 model weights (this will take time on first run)...")\n    # Load in float16 to save memory and speed up inference on GPUs like T4\n    model = Blip2ForConditionalGeneration.from_pretrained(\n        model_name, \n        torch_dtype=torch.float16,\n        use_safetensors=True\n    ).to(device)\n    \n    return processor, model, device\n\ndef generate_captions(image_paths, processor, model, device):\n    """\n    Generate descriptive captions for a list of images using BLIP-2.\n    """\n    valid_paths = [p for p in image_paths if os.path.exists(p)]\n    captions_dict = {}\n    \n    if not valid_paths:\n        return captions_dict\n\n    # Process images one by one to avoid VRAM OOM errors during text generation\n    for path in valid_paths:\n        try:\n            raw_image = Image.open(path).convert("RGB")\n            \n            # Không sử dụng prompt rườm rà vì lõi OPT-2.7B dễ bị "ảo giác" (hallucination)\n            # Dùng prompt hỏi đáp chuẩn của BLIP-2 hoặc sinh tự do\n            prompt = "Question: describe this image in detail. Answer:"\n            inputs = processor(raw_image, text=prompt, return_tensors="pt").to(device, torch.float16)\n            \n            # Thêm repetition_penalty để chống lặp từ (như [dog][dog]...)\n            # Thêm length_penalty và giới hạn token để câu văn gọn gàng\n            generated_ids = model.generate(\n                **inputs, \n                max_new_tokens=30,\n                min_new_tokens=5,\n                repetition_penalty=1.5\n            )\n            generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()\n            \n            # Xóa câu prompt mồi ra khỏi kết quả (do kiến trúc Decoder-only của OPT sẽ trả về cả prompt)\n            if generated_text.startswith(prompt):\n                generated_text = generated_text[len(prompt):].strip()\n            \n            captions_dict[path] = generated_text\n        except Exception as e:\n            print(f"Error generating caption for {path}: {e}")\n            captions_dict[path] = ""\n            \n    return captions_dict\n\ndef evaluate_clip_score(captions_dict):\n    """\n    Reference-free Benchmark: CLIP-Score.\n    Measures how well the generated text matches the visual content of the image.\n    Score ranges roughly from 20 to 40. Higher is better.\n    """\n    if not captions_dict:\n        return 0.0\n        \n    print("Loading CLIP for benchmark scoring...")\n    try:\n        clip_processor, clip_model, device = load_clip_model()\n    except Exception as e:\n        print(f"Failed to load CLIP for evaluation: {e}")\n        return 0.0\n        \n    total_score = 0.0\n    valid_count = 0\n    \n    with torch.no_grad():\n        for img_path, text in captions_dict.items():\n            if not text or not os.path.exists(img_path):\n                continue\n                \n            image = Image.open(img_path).convert("RGB")\n            \n            # Pass both image and generated text to CLIP\n            inputs = clip_processor(text=[text], images=image, return_tensors="pt", padding=True).to(device)\n            outputs = clip_model(**inputs)\n            \n            # Extract logits (similar to unscaled cosine similarity)\n            logits_per_image = outputs.logits_per_image  # shape: [1, 1]\n            score = logits_per_image.item()\n            \n            # Typical CLIP score is scaled by a factor (usually 2.5 in academic papers) for readability\n            scaled_score = max(0.0, score * 2.5)\n            \n            total_score += scaled_score\n            valid_count += 1\n            \n    if valid_count == 0:\n        return 0.0\n        \n    return total_score / valid_count\n\nif __name__ == "__main__":\n    import glob\n    \n    # Use relative paths\n    video_name = "0tmA_C6XwfM"\n    keyframe_dir = os.path.join("keyframes", video_name)\n    image_paths = sorted(glob.glob(os.path.join(keyframe_dir, "*.jpg")))\n    \n    # For testing, we only take the first 3 images so we don\'t wait forever\n    test_paths = image_paths[:3]\n    \n    if not test_paths:\n        print(f"Error: No images found in {keyframe_dir}")\n        exit(1)\n        \n    try:\n        print(f"Initializing BLIP-2 pipeline for {len(test_paths)} test images...")\n        processor, model, device = load_blip2_model()\n        print(f"BLIP-2 loaded on {device}")\n        \n        print("Generating captions...")\n        results = generate_captions(test_paths, processor, model, device)\n        \n        for path, caption in results.items():\n            print(f"Image: {os.path.basename(path)} -> Caption: {caption}")\n            \n        # Run Reference-free benchmark\n        print("\\\\nCalculating CLIP-Score...")\n        avg_clip_score = evaluate_clip_score(results)\n        \n        print("-" * 40)\n        print("BLIP-2 CAPTIONING BENCHMARK RESULTS")\n        print("-" * 40)\n        print(f"Average CLIP-Score: {avg_clip_score:.2f}")\n        \n        if avg_clip_score > 70.0:\n            print("Evaluation: Excellent. High correlation between text and image.")\n        elif avg_clip_score > 60.0:\n            print("Evaluation: Good. The captions accurately describe the scenes.")\n        else:\n            print("Evaluation: Average. Captions might lack specific details.")\n            \n    except Exception as e:\n        print(f"Execution error: {e}")\n',
    'main_visual.py': 'import os\nimport json\nimport argparse\n\n# Import modules from our visual pipeline\nfrom scene_detector import extract_scenes_and_keyframes, evaluate_scene_diversity\nfrom clip_filter import load_clip_model, extract_image_embeddings, filter_keyframes, evaluate_filter_quality\nfrom image_captioner import load_blip2_model, generate_captions, evaluate_clip_score\n\ndef run_visual_pipeline(video_path, output_dir="output", keep_ratio=0.7):\n    """\n    End-to-End Visual Pipeline:\n    Video -> PySceneDetect -> CLIP Filter -> BLIP-2 Captions\n    """\n    print("=" * 60)\n    print(f"STARTING VISUAL PIPELINE FOR: {video_path}")\n    print("=" * 60)\n    \n    video_name = os.path.splitext(os.path.basename(video_path))[0]\n    keyframe_dir = os.path.join(output_dir, "keyframes")\n    result_dir = os.path.join(output_dir, "results")\n    os.makedirs(result_dir, exist_ok=True)\n    \n    # ---------------------------------------------------------\n    # STEP 1: SCENE DETECTION\n    # ---------------------------------------------------------\n    print("\\\\n[STEP 1] Running Scene Detection")\n    metadata = extract_scenes_and_keyframes(video_path, output_dir=keyframe_dir, threshold=27.0)\n    \n    if not metadata:\n        print("Error: No scenes detected.")\n        return None\n        \n    diversity_score = evaluate_scene_diversity(metadata)\n    print(f"-> Extracted {len(metadata)} keyframes.")\n    print(f"-> Scene Diversity Score: {diversity_score:.4f}")\n    \n    # Get paths of extracted images\n    image_paths = [m[\'keyframe_path\'] for m in metadata if os.path.exists(m[\'keyframe_path\'])]\n    \n    # ---------------------------------------------------------\n    # STEP 2: CLIP FILTERING\n    # ---------------------------------------------------------\n    print("\\\\n[STEP 2] Running Semantic Filtering (CLIP + KMeans)...")\n    clip_processor, clip_model, clip_device = load_clip_model()\n    \n    embeddings, valid_paths = extract_image_embeddings(image_paths, clip_processor, clip_model, clip_device)\n    filtered_paths, filtered_embeddings = filter_keyframes(valid_paths, embeddings, keep_ratio=keep_ratio)\n    \n    orig_dist, filt_dist = evaluate_filter_quality(embeddings, filtered_embeddings)\n    print(f"-> Original frames: {len(valid_paths)} | Filtered frames: {len(filtered_paths)}")\n    print(f"-> Semantic distance improved: {orig_dist:.4f} -> {filt_dist:.4f}")\n    \n    # ---------------------------------------------------------\n    # STEP 3: BLIP-2 IMAGE CAPTIONING\n    # ---------------------------------------------------------\n    print("\\\\n[STEP 3] Running Image Captioning (BLIP-2)...")\n    blip_processor, blip_model, blip_device = load_blip2_model()\n    \n    captions_dict = generate_captions(filtered_paths, blip_processor, blip_model, blip_device)\n    \n    print("\\\\n[STEP 4] Evaluating Captions (CLIP-Score)...")\n    avg_clip_score = evaluate_clip_score(captions_dict)\n    print(f"-> Average CLIP-Score: {avg_clip_score:.2f}")\n    \n    # ---------------------------------------------------------\n    # FINAL: COMPILE AND SAVE RESULTS\n    # ---------------------------------------------------------\n    print("\\\\n[FINAL] Compiling narrative summary...")\n    \n    summary_lines = []\n    for path in filtered_paths:\n        if path in captions_dict:\n            filename = os.path.basename(path)\n            caption = captions_dict[path]\n            summary_lines.append(f"[{filename}]: {caption}")\n            \n    final_text = " ".join([captions_dict[p] for p in filtered_paths if p in captions_dict])\n    \n    # Save to JSON\n    output_json = os.path.join(result_dir, f"{video_name}_summary.json")\n    with open(output_json, \'w\', encoding=\'utf-8\') as f:\n        json.dump({\n            "video_name": video_name,\n            "metrics": {\n                "diversity_score": diversity_score,\n                "clip_distance_improvement": f"{orig_dist:.4f} -> {filt_dist:.4f}",\n                "clip_score": avg_clip_score\n            },\n            "captions": summary_lines,\n            "combined_text": final_text\n        }, f, indent=4)\n        \n    # Save raw text for the LLM step\n    output_txt = os.path.join(result_dir, f"{video_name}_summary.txt")\n    with open(output_txt, \'w\', encoding=\'utf-8\') as f:\n        f.write(final_text)\n        \n    print(f"\\\\nPipeline completed successfully!")\n    print(f"Results saved to: {output_json} and {output_txt}")\n    \n    return final_text\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description="Run the full Video Visual Summarization Pipeline.")\n    parser.add_argument("--video", type=str, default=os.path.join("..", "dataset", "Visual", "0tmA_C6XwfM.mp4"),\n                        help="Path to the input video file")\n    parser.add_argument("--out_dir", type=str, default="pipeline_output",\n                        help="Directory to save output files")\n    parser.add_argument("--ratio", type=float, default=0.7,\n                        help="Ratio of frames to keep after CLIP filtering (0.0 to 1.0)")\n                        \n    args = parser.parse_args()\n    \n    run_visual_pipeline(args.video, output_dir=args.out_dir, keep_ratio=args.ratio)\n',
}

for name, code in files.items():
    (PKG / name).write_text(code, encoding='utf-8')

if str(PKG) not in sys.path:
    sys.path.insert(0, str(PKG))
print('Da tao package tai:', PKG)

Da tao package tai: /kaggle/working/visual


In [3]:
!pip install scenedetect[opencv] transformers accelerate bitsandbytes scikit-learn pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.7/134.7 kB 5.4 MB/s eta 0:00:00


In [4]:
from main_visual import run_visual_pipeline
import os

DATASET_DIR = "/kaggle/input/datasets/duc63minh/tvsum-visual"
VIDEO_PATH = os.path.join(DATASET_DIR, "0tmA_C6XwfM.mp4")

print(f"Đang kiểm tra sự tồn tại của file: {VIDEO_PATH}")
if not os.path.exists(VIDEO_PATH):
    print("LỖI: Không tìm thấy file video. Vui lòng kiểm tra lại cấu trúc thư mục của Dataset trên Kaggle!")
else:
    # Khởi chạy Pipeline nếu tìm thấy file
    summary_text = run_visual_pipeline(VIDEO_PATH, output_dir="/kaggle/working/output", keep_ratio=0.7)
    
    print("\n==== VĂN BẢN TÓM TẮT TỪ HÌNH ẢNH ====\n")
    print(summary_text)

/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"


Đang kiểm tra sự tồn tại của file: /kaggle/input/datasets/duc63minh/tvsum-visual/0tmA_C6XwfM.mp4
STARTING VISUAL PIPELINE FOR: /kaggle/input/datasets/duc63minh/tvsum-visual/0tmA_C6XwfM.mp4
\n[STEP 1] Running Scene Detection
Starting video analysis: /kaggle/input/datasets/duc63minh/tvsum-visual/0tmA_C6XwfM.mp4
Number of detected scenes: 20
Keyframe extraction completed.
-> Extracted 20 keyframes.
-> Scene Diversity Score: 0.5857
\n[STEP 2] Running Semantic Filtering (CLIP + KMeans)...


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

\n[STEP 4] Evaluating Captions (CLIP-Score)...
Loading CLIP for benchmark scoring...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


-> Average CLIP-Score: 72.55
\n[FINAL] Compiling narrative summary...
\nPipeline completed successfully!
Results saved to: /kaggle/working/output/results/0tmA_C6XwfM_summary.json and /kaggle/working/output/results/0tmA_C6XwfM_summary.txt

==== VĂN BẢN TÓM TẮT TỪ HÌNH ẢNH ====

a cute little dog is sitting on the ground with its head down a woman with her hands on the table a man sitting on the couch with his dog a dog with its head on the lap of his owner a man sitting on the floor with his dog a man is petting his dog a man is playing with his dog a dog's paw is being scratched by someone a dog's paw is being trimmed a man is petting his dog a dog playing with his toy the scissors are brown and have a white handle a white dog with red shoes a woman with dark hair and blue shirt
